# Step 7 - Comparing Generated Features to the Author's Reference

This step validates a generated `.npz` batch against one of the author's reference batches.

The two files contain **different video clips**, and the features are produced by **different
CNN weights** (torchvision ImageNet VGG-16 here, versus the author's Caffe VGG with a different
object detector). Element-wise equality is therefore neither expected nor meaningful. Instead,
faithfulness is assessed on two levels:

1. **Format** - identical keys, shapes, dtypes, and label structure.
2. **Statistical character** - non-negativity, value range, sparsity, and objects-per-frame.

> A fair comparison requires a generated batch built with `MAX_FRAMES = 100`. A batch built
> with fewer frames will appear almost entirely zero and will not be representative.

## 1. Load both batches

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

AUTHOR_PATH    = '../Dataset/features/test/batch_001.npz'
GENERATED_PATH = '../Dataset/features_generated/test/batch_001.npz'

for p in (AUTHOR_PATH, GENERATED_PATH):
    print(('OK  ' if os.path.exists(p) else 'MISSING '), p)

author = np.load(AUTHOR_PATH)
gen    = np.load(GENERATED_PATH)

## 2. Format comparison
Keys, array shapes, and dtypes should match exactly.

In [ ]:
def fmt(f):
    return {k: (f[k].shape, str(f[k].dtype)) for k in f.files}

fa, fg = fmt(author), fmt(gen)
print('%-8s %-28s %-28s %s' % ('key', 'author', 'generated', 'match'))
print('-'*80)
for k in sorted(set(fa) | set(fg)):
    a = fa.get(k, '-'); g = fg.get(k, '-')
    print('%-8s %-28s %-28s %s' % (k, a, g, 'YES' if a == g else 'NO'))

## 3. Label structure
Both should be one-hot `(10, 2)` with a 5 accident / 5 non-accident composition.

In [ ]:
def describe_labels(f):
    lab = f['labels']
    order = ''.join('A' if r[1] == 1 else 'n' for r in lab)
    return order, order.count('A'), order.count('n')

for name, f in [('author', author), ('generated', gen)]:
    order, a, n = describe_labels(f)
    print(f'{name:10s}: {a} accident + {n} non   order={order}')

## 4. Value statistics
Both feature tensors should be non-negative (post-ReLU) and sparse.

In [ ]:
def stats(f):
    d = f['data'].astype(np.float32)
    return {
        'min':      float(d.min()),
        'max':      float(d.max()),
        'mean':     float(d.mean()),
        'std':      float(d.std()),
        'frac_zero':float((d == 0).mean()),
        'all_>=0':  bool((d >= 0).all()),
    }

sa, sg = stats(author), stats(gen)
print('%-10s %-14s %-14s' % ('stat', 'author', 'generated'))
print('-'*40)
for k in sa:
    va, vg = sa[k], sg[k]
    va = f'{va:.4f}' if isinstance(va, float) else str(va)
    vg = f'{vg:.4f}' if isinstance(vg, float) else str(vg)
    print('%-10s %-14s %-14s' % (k, va, vg))

## 5. Distribution of feature values
Histogram of non-zero feature values. The exact shapes differ (different CNN weights), but
both should be positive and right-skewed.

In [ ]:
def sample_nonzero(f, n=100000):
    d = f['data'].astype(np.float32).ravel()
    d = d[d > 0]
    if d.size > n:
        d = np.random.choice(d, n, replace=False)
    return d

va, vg = sample_nonzero(author), sample_nonzero(gen)
plt.figure(figsize=(11, 4))
plt.hist(va, bins=60, alpha=0.6, density=True, label='author')
plt.hist(vg, bins=60, alpha=0.6, density=True, label='generated')
plt.title('Distribution of non-zero feature values')
plt.xlabel('feature value'); plt.ylabel('density'); plt.legend()
plt.show()

## 6. Objects per frame
The number of filled slots per frame reflects detector behaviour. Distributions should be
broadly similar (whole frame plus a few objects per frame).

In [ ]:
def slots_per_frame(f):
    d = f['data'].astype(np.float32)
    occ = (np.abs(d).sum(axis=3) > 0)        # (videos, 100, 20)
    return occ.sum(axis=2).ravel()           # slots filled per (video, frame)

spa, spg = slots_per_frame(author), slots_per_frame(gen)
print('mean slots/frame  -> author: %.2f   generated: %.2f' % (spa.mean(), spg.mean()))

bins = np.arange(0, 21) - 0.5
plt.figure(figsize=(11, 4))
plt.hist(spa, bins=bins, alpha=0.6, density=True, label='author')
plt.hist(spg, bins=bins, alpha=0.6, density=True, label='generated')
plt.title('Slots filled per frame'); plt.xlabel('slots filled'); plt.ylabel('density')
plt.legend(); plt.show()

## Summary

Faithfulness is evaluated at two levels:

- **Format** (keys, shapes, dtypes, label structure): expected to match exactly. This confirms
  the generated files are drop-in compatible with the author's model.
- **Statistical character** (non-negativity, sparsity, value range, objects per frame):
  expected to be similar but not identical.

Exact feature values differ because the clips differ and the CNN/detector weights differ
(torchvision ImageNet VGG-16 versus the author's Caffe VGG). This is an inherent limitation of
reproducing an unreleased feature-extraction stage and does not affect format compatibility.